# imports

In [ ]:
import sys
project_root = r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/10_code/UTvsXCT-preprocessing'
if project_root not in sys.path:
    sys.path.append(project_root)
import importlib
from preprocess_tools import onlypores, datasetmaker, io, aligner, register
datasetmaker = importlib.reload(datasetmaker)
import numpy as np
import pandas as pd
import tifffile
from dbtools import dbtools as db
from dbtools import load as load
from pathlib import Path
import ast

# Path mapping

In [ ]:
LOCAL_DATA_ROOT = Path('/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente')
UNC_DATA_ROOT = r'\\192.168.10.106\imdea\DataDriven_UT_AlbertoVicente'


def resolve_data_path(path, must_exist=True):
    path_str = str(path).strip()
    direct_path = Path(path_str)

    if direct_path.exists():
        return direct_path

    normalized_path = path_str.replace('\\', '/')
    normalized_unc_root = UNC_DATA_ROOT.replace('\\', '/')

    if normalized_path.lower().startswith(normalized_unc_root.lower()):
        relative_path = normalized_path[len(normalized_unc_root):].lstrip('/')
        local_path = LOCAL_DATA_ROOT / relative_path
        if local_path.exists() or not must_exist:
            return local_path

    if must_exist:
        raise FileNotFoundError(
            f"Could not resolve data path: {path_str}\n"
            f"Tried direct path and UNC mapping to: {LOCAL_DATA_ROOT}"
        )

    return direct_path

# Database connection

In [ ]:
try:
    conn = db.connect()
    print("Connected to the database")
except Exception as error:
    print(error)

# Select measurements to create the image datasets

## Measurement type

In [ ]:
measurementtypes_table = db.get_data_metadata('measurementtypes')

measurementtypes_table

In [ ]:
ut_type = 10

xct_type = 6

## Selecting UT measurements

In [ ]:
ut_measurements_table = db.get_data_metadata('measurements')

ut_measurements_table = ut_measurements_table[ut_measurements_table['measurementtype_id_measurement'] == ut_type]

ut_measurements_table

## Selecting XCT measurements

In [ ]:
xct_measurements_table = db.get_data_metadata('measurements')

xct_measurements_table = xct_measurements_table[xct_measurements_table['measurementtype_id_measurement'] == xct_type]

xct_measurements_table

## Getting registered pairs

In [ ]:
registrations_table = db.get_data_metadata('registrations')

registrations_table = registrations_table[registrations_table['reference_measurement_id_registration'].isin(ut_measurements_table['id_measurement'])]

registrations_table = registrations_table[registrations_table['registered_measurement_id_registration'].isin(xct_measurements_table['id_measurement'])]

registrations_table

## Sample and panel metadata

In [ ]:
sample_measurements_table = db.relation_metadata('samples', 'measurements', 'sample_measurements')
samples_table = db.get_data_metadata('samples')
panels_table = db.get_data_metadata('panels')

sample_measurements_table

In [ ]:
def lookup_sample_metadata(measurement_id):
    sample_rows = sample_measurements_table[sample_measurements_table['id_measurement'] == measurement_id]
    if sample_rows.empty:
        return {
            'sample_id': None,
            'sample_name': None,
            'panel_id': None,
            'panel_name': None,
        }

    sample_row = sample_rows.iloc[0]
    sample_id = sample_row.get('id_sample')
    sample_name = sample_row.get('name_sample')
    panel_id = sample_row.get('panel_id_sample')

    if pd.isna(panel_id) and sample_id is not None:
        matching_samples = samples_table[samples_table['id_sample'] == sample_id]
        if not matching_samples.empty:
            panel_id = matching_samples.iloc[0].get('panel_id_sample')

    panel_name = None
    if panel_id is not None and not pd.isna(panel_id):
        matching_panels = panels_table[panels_table['id_panel'] == panel_id]
        if not matching_panels.empty:
            panel_name = matching_panels.iloc[0].get('name_panel')

    return {
        'sample_id': sample_id,
        'sample_name': sample_name,
        'panel_id': panel_id,
        'panel_name': panel_name,
    }


registered_pairs = []

for _, registration in registrations_table.iterrows():
    reference_id = registration['reference_measurement_id_registration']
    registered_id = registration['registered_measurement_id_registration']

    reference_measurement = ut_measurements_table[ut_measurements_table['id_measurement'] == reference_id].iloc[0]
    registered_measurement = xct_measurements_table[xct_measurements_table['id_measurement'] == registered_id].iloc[0]
    sample_metadata = lookup_sample_metadata(reference_id)

    registered_pairs.append({
        'registration_id': registration['id_registration'],
        'reference_measurement_id': reference_id,
        'registered_measurement_id': registered_id,
        'reference_measurement_path': reference_measurement['file_path_measurement'],
        'registered_measurement_path': registered_measurement['file_path_measurement'],
        'sample_id': sample_metadata['sample_id'],
        'sample_name': sample_metadata['sample_name'],
        'panel_id': sample_metadata['panel_id'],
        'panel_name': sample_metadata['panel_name'],
    })

registered_pairs_df = pd.DataFrame(registered_pairs)

print(f"Found {len(registered_pairs_df)} registrations")
registered_pairs_df

# Dataset type selection

In [ ]:
datasettype_table = db.get_data('datasettypes')

datasettype_table

In [ ]:
# Set to the appropriate datasettype id for conical image datasets.
# Set to None to skip DB tracking (useful when no dedicated type exists yet).
datasettype = None
load_into_database = False

# Discard already computed datasets

In [ ]:
if datasettype is not None:
    try:
        dataset_registrations_table = db.relation_metadata('datasets', 'registrations', 'dataset_registrations')

        dataset_registrations_table = dataset_registrations_table[dataset_registrations_table['datasettype_id_dataset'] == datasettype]

        dataset_registrations = dataset_registrations_table['id_registration'].values

    except Exception as e:
        print("No dataset registrations found or error occurred:", e)
        dataset_registrations = []
else:
    dataset_registrations = []

# Saving folder

In [ ]:
folder = Path(r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Airbus/Hexcel/2026/images_conical')

folder.mkdir(parents=True, exist_ok=True)

# Image settings

In [ ]:
material_threshold = 0.8
stop_after_first = True

# Conical beam parameters

Beam radii from the 5 MHz Technisonic ISL-0503-HR transducer characterisation
(focal length 76.2 mm, standoff 67.4 mm). See wiki: conical-patches-xct.

Recalculate if the transducer or standoff distance changes.

In [ ]:
# Half of the beam diameter at each face of the specimen
radius_front_mm = 1.615  # 3.23 mm beam width at frontwall / 2
radius_back_mm  = 1.40   # 2.80 mm beam width at backwall  / 2

print(f"Beam radius at frontwall : {radius_front_mm} mm  ({radius_front_mm / 0.025:.0f} XCT pixels)")
print(f"Beam radius at backwall  : {radius_back_mm} mm  ({radius_back_mm  / 0.025:.0f} XCT pixels)")

# Resolutions

In [ ]:
xct_resolution = float(measurementtypes_table[measurementtypes_table['id_measurementtype'] == xct_type]['voxel_size_measurementtype'].values[0].split(' ')[0])
ut_resolution = float(measurementtypes_table[measurementtypes_table['id_measurementtype'] == ut_type]['x_resolution_measurementtype'].values[0].split(' ')[0])
print(f"XCT resolution: {xct_resolution}, UT resolution: {ut_resolution}")

# Dataset generation

In [ ]:
excluded_samples = ['1.27', 'Na_02_1', 'Na_02_2', 'Na_02_3', 'Na_02_4', 'Na_02_5', 'Na_09_5', 'Na_10_1', 'Na_10_2', 'Na_10_3']

manifest_rows = []

for _, pair in registered_pairs_df.iterrows():
    reference_measurement_path_raw = pair['reference_measurement_path']
    registered_measurement_path_raw = pair['registered_measurement_path']
    registration_id = pair['registration_id']
    sample_name = pair['sample_name']
    sample_folder_name = sample_name if pd.notna(sample_name) else f"registration_{registration_id}"

    if sample_name in excluded_samples:
        print(f"Sample {sample_name} is in the exclusion list, skipping...")
        continue

    if registration_id in dataset_registrations:
        print(f"Dataset for registration {registration_id} already exists, skipping...")
        continue

    reference_measurement_path = resolve_data_path(reference_measurement_path_raw)
    registered_measurement_path = resolve_data_path(registered_measurement_path_raw)

    print(f"Creating conical image dataset for registration {registration_id} — {sample_name}")

    ut_volume = io.load_tif(reference_measurement_path)
    xct_volume = io.load_tif(registered_measurement_path)

    xct_volume = np.transpose(xct_volume, (1, 2, 0))
    ut_volume = np.transpose(ut_volume, (1, 2, 0))

    registration_parameters = registrations_table[registrations_table['id_registration'] == registration_id]['registration_matrix_registration'].values[0]
    registration_parameters = np.array(ast.literal_eval(registration_parameters))

    xct_volume = register.apply_registration(ut_volume, xct_volume, registration_parameters, ut_resolution, xct_resolution, parallel=True)

    _, frontwall, backwall = aligner.crop_walls(xct_volume)

    xct_volume = np.transpose(xct_volume, (2, 0, 1))
    ut_volume = np.transpose(ut_volume, (2, 0, 1))

    onlypores_volume, material_mask, _ = onlypores.onlypores(xct_volume, frontwall, backwall, min_size_filtering=8)

    dataset_folder = folder / f"{sample_folder_name}" / f"registration_{registration_id}"
    dataset_folder.mkdir(parents=True, exist_ok=True)

    ut_image, volfrac_maps, areafrac_image, depth_rois = datasetmaker.main_images(
        onlypores_volume,
        material_mask,
        ut_volume,
        xct_resolution,
        ut_resolution,
        material_threshold=material_threshold,
        conical=True,
        radius_front_mm=radius_front_mm,
        radius_back_mm=radius_back_mm,
    )

    depth_roi_regions = tuple(roi['region'] for roi in depth_rois)
    depth_roi_start_from_frontwall = tuple(roi['start_from_frontwall'] for roi in depth_rois)
    depth_roi_end_exclusive_from_frontwall = tuple(roi['end_exclusive_from_frontwall'] for roi in depth_rois)
    depth_roi_end_inclusive_from_frontwall = tuple(roi['end_inclusive_from_frontwall'] for roi in depth_rois)
    depth_roi_start_z = tuple(roi['start_z'] for roi in depth_rois)
    depth_roi_end_exclusive_z = tuple(roi['end_exclusive_z'] for roi in depth_rois)
    depth_roi_end_inclusive_z = tuple(roi['end_inclusive_z'] for roi in depth_rois)
    material_frontwall_z = depth_rois[0]['material_frontwall_z']
    material_backwall_z = depth_rois[0]['material_backwall_z']

    image_outputs = {
        'ut': (dataset_folder / 'ut_volume.tif', ut_image, None),
        'volfrac': (dataset_folder / 'volfrac_maps.tif', volfrac_maps.astype(np.float32), depth_roi_regions),
        'areafrac': (dataset_folder / 'areafrac_map.tif', areafrac_image.astype(np.float32), None),
    }

    current_rows = []
    for image_kind, (image_path, image_array, depth_region) in image_outputs.items():
        tifffile.imwrite(image_path, image_array, bigtiff=True)
        current_rows.append({
            'image_kind': image_kind,
            'image_path': str(image_path),
            'image_shape': tuple(image_array.shape),
            'image_dtype': str(image_array.dtype),
            'file_format': 'tif',
            'depth_region': depth_region,
            'sample_id': pair['sample_id'],
            'sample_name': sample_name,
            'panel_id': pair['panel_id'],
            'panel_name': pair['panel_name'],
            'registration_id': registration_id,
            'reference_measurement_id': pair['reference_measurement_id'],
            'registered_measurement_id': pair['registered_measurement_id'],
            'reference_measurement_path': str(reference_measurement_path),
            'registered_measurement_path': str(registered_measurement_path),
            'reference_measurement_path_raw': reference_measurement_path_raw,
            'registered_measurement_path_raw': registered_measurement_path_raw,
            'ut_image_shape': tuple(ut_image.shape),
            'target_map_shape': tuple(volfrac_maps.shape[1:]),
            'volfrac_maps_shape': tuple(volfrac_maps.shape),
            'volfrac_region_names': depth_roi_regions,
            'depth_roi_start_from_frontwall': depth_roi_start_from_frontwall,
            'depth_roi_end_exclusive_from_frontwall': depth_roi_end_exclusive_from_frontwall,
            'depth_roi_end_inclusive_from_frontwall': depth_roi_end_inclusive_from_frontwall,
            'depth_roi_start_z': depth_roi_start_z,
            'depth_roi_end_exclusive_z': depth_roi_end_exclusive_z,
            'depth_roi_end_inclusive_z': depth_roi_end_inclusive_z,
            'material_frontwall_z': material_frontwall_z,
            'material_backwall_z': material_backwall_z,
            'material_threshold': material_threshold,
            'xct_resolution': xct_resolution,
            'ut_resolution': ut_resolution,
            'conical': True,
            'radius_front_mm': radius_front_mm,
            'radius_back_mm': radius_back_mm,
        })

    registration_manifest_df = pd.DataFrame(current_rows)
    registration_manifest_path = dataset_folder / 'image_manifest.csv'
    registration_manifest_df.to_csv(registration_manifest_path, index=False)
    manifest_rows.extend(current_rows)

    if load_into_database and datasettype is not None:
        rows = int(np.prod(volfrac_maps.shape))
        targets = ['volfrac_front', 'volfrac_middle', 'volfrac_back', 'areafrac']
        description = (
            'Conical image-map dataset. VVF computed via FFT disk convolution '
            f'(radius_front={radius_front_mm} mm, radius_back={radius_back_mm} mm). '
            'Created with preprocess_tools datasetmaker.main_images(conical=True).'
        )

        load.load_dataset(
            conn,
            datasettype_id=datasettype,
            file_path=str(registration_manifest_path),
            rows=rows,
            patch_size='no_ut_patches',
            targets=targets,
            reconstruction_shape=volfrac_maps.shape,
            registration_ids=[registration_id],
            description=description,
        )

    print(f"  Saved to {dataset_folder}")

    if stop_after_first:
        break

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = folder / 'image_dataset_manifest.csv'
manifest_df.to_csv(manifest_path, index=False)

print(f"Saved image dataset manifest to {manifest_path}")
manifest_df